# Step 0. Install Libraries and Setup

Install necessary libraries for VideoMAE and video processing.

In [1]:
!pip install decord seaborn transformers evaluate accelerate ffmpeg-python av scikit-learn pandas pytorchvideo comet_ml -q
!pip install --upgrade --no-cache-dir gdown -q
!pip install comet-ml>=3.43.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.3/782.3 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s eta 0:00:00


In [2]:
import comet_ml
comet_ml.init(api_key="VDWJEDjieOQbqDskMAw7Cr1av", project_name="gym-pose-videomae")
import torchvision.transforms.functional as F_t
import sys
from tqdm.auto import tqdm
from transformers import EarlyStoppingCallback

# This creates a dummy module so the old import path works
sys.modules["torchvision.transforms.functional_tensor"] = F_t

COMET WARNING: comet_ml.init() is deprecated and will be removed soon. Please use comet_ml.login()
COMET INFO: Valid Comet API Key saved in /root/.comet.config (set COMET_CONFIG to change where it is saved).


# Step 1. Data Preparation

Scan the dataset directory, split into Training and Validation sets, and generate TSV files required for training.

In [3]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Define Dataset Path
DATA_ROOT_DIR = '/kaggle/input/workoutfitness-video'

# Label Mapping
LABEL2ID = {
    'barbell biceps curl': 0,
    'bench press': 1,
    'chest fly machine': 2,
    'deadlift': 3,
    'decline bench press': 4,
    'hammer curl': 5,
    'hip thrust': 6,
    'incline bench press': 7,
    'lat pulldown': 8,
    'lateral raise': 9,
    'leg extension': 10,
    'leg raises': 11,
    'plank': 12,
    'pull Up': 13,
    'push-up': 14,
    'romanian deadlift': 15,
    'russian twist': 16,
    'shoulder press': 17,
    'squat': 18,
    't bar row': 19,
    'tricep Pushdown': 20,
    'tricep dips': 21,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Scan Directory
data_entries = []
for label_name in tqdm(os.listdir(DATA_ROOT_DIR), desc="Scanning Classes"):
    class_dir = os.path.join(DATA_ROOT_DIR, label_name)
    if os.path.isdir(class_dir) and label_name in LABEL2ID:
        label_id = LABEL2ID[label_name]
        for video_file in os.listdir(class_dir):
             if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                video_path = os.path.join(class_dir, video_file)
                data_entries.append({'path': video_path, 'label_id': label_id})

df = pd.DataFrame(data_entries)
print(f"Found {len(df)} total videos.")

# Split Data
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_id'])

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")

Scanning Classes:   0%|          | 0/22 [00:00<?, ?it/s]

Found 652 total videos.
Training samples: 521
Validation samples: 131


# Step 2. Model Training

Defines the training pipeline, helper functions, and starts the training process.

In [4]:
import gc
from transformers import TrainerCallback
import evaluate
import numpy as np
import pytorchvideo.data
import torch
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    Normalize,
    RandomShortSideScale,
    UniformTemporalSubsample,
)
from torchvision.transforms import (
    Compose,
    Lambda,
    RandomCrop,
    RandomHorizontalFlip,
    Resize,
)
from transformers import (
    Trainer,
    TrainingArguments,
    VideoMAEForVideoClassification,
    VideoMAEImageProcessor,
)

# --- Helper Functions ---

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions."""
    metric = evaluate.load("accuracy")
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

def collate_fn(examples):
    """The collation function to be used by `Trainer` to prepare data batches."""
    pixel_values = torch.stack(
        [example["video"].permute(1, 0, 2, 3) for example in examples]
    )
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

def prepare_dataset(train_df, val_df, image_processor, model, sample_rate=4, fps=30):
    mean = image_processor.image_mean
    std = image_processor.image_std
    if "shortest_edge" in image_processor.size:
        height = width = image_processor.size["shortest_edge"]
    else:
        height = image_processor.size["height"]
        width = image_processor.size["width"]
    resize_to = (height, width)

    num_frames_to_sample = model.config.num_frames
    clip_duration = num_frames_to_sample * sample_rate / fps

    train_transform = Compose(
        [
            ApplyTransformToKey(
                key="video",
                transform=Compose(
                    [
                        UniformTemporalSubsample(num_frames_to_sample),
                        Lambda(lambda x: x / 255.0),
                        Normalize(mean, std),
                        RandomShortSideScale(min_size=256, max_size=320),
                        RandomCrop(resize_to),
                        RandomHorizontalFlip(p=0.5),
                    ]
                ),
            ),
        ]
    )

    val_transform = Compose(
        [
            ApplyTransformToKey(
                key="video",
                transform=Compose(
                    [
                        UniformTemporalSubsample(num_frames_to_sample),
                        Lambda(lambda x: x / 255.0),
                        Normalize(mean, std),
                        Resize(resize_to),
                    ]
                ),
            ),
        ]
    )

    train_labeled_video_paths = [
        (row['path'], {'label': row['label_id']}) 
        for _, row in train_df.iterrows()
    ]
    
    val_labeled_video_paths = [
        (row['path'], {'label': row['label_id']}) 
        for _, row in val_df.iterrows()
    ]

    train_dataset = pytorchvideo.data.LabeledVideoDataset(
        labeled_video_paths=train_labeled_video_paths,
        clip_sampler=pytorchvideo.data.make_clip_sampler("random", clip_duration),
        decode_audio=False,
        transform=train_transform,
    )

    val_dataset = pytorchvideo.data.LabeledVideoDataset(
        labeled_video_paths=val_labeled_video_paths,
        clip_sampler=pytorchvideo.data.make_clip_sampler("uniform", clip_duration),
        decode_audio=False,
        transform=val_transform,
    )

    return train_dataset, val_dataset


2026-02-25 20:13:40.723506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772050420.940228      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772050421.004579      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772050421.577640      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772050421.577693      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772050421.577697      25 computation_placer.cc:177] computation placer alr

In [5]:

class MemoryCleanupCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        gc.collect()
        torch.cuda.empty_cache()

def train_model(output_dir, model, image_processor, train_dataset, val_dataset, batch_size=4, num_epochs=10, learning_rate=5e-5, warmup_ratio=0.1, num_workers=4):
    args = TrainingArguments(output_dir, remove_unused_columns=False, eval_strategy="epoch", save_strategy="epoch", learning_rate=learning_rate,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size, warmup_ratio=warmup_ratio, logging_steps=10, load_best_model_at_end=True,
        metric_for_best_model="accuracy", max_steps=(train_dataset.num_videos // batch_size) * num_epochs, report_to=["comet_ml"],
        fp16=True, dataloader_num_workers=num_workers, dataloader_persistent_workers=True if num_workers > 0 else False,
        gradient_checkpointing=True, lr_scheduler_type="cosine"
    )

    trainer = Trainer(model, args, train_dataset=train_dataset, eval_dataset=val_dataset, processing_class=image_processor, compute_metrics=compute_metrics,
        data_collator=collate_fn, callbacks=[EarlyStoppingCallback(early_stopping_patience=3), MemoryCleanupCallback()])

    trainer.train()
    return trainer

# Setup Model and Run Training

# Use 'MCG-NJU/videomae-large-finetuned-kinetics' if you have enough VRAM (24GB+)
MODEL_CKPT = "MCG-NJU/videomae-base-finetuned-kinetics" 
OUTPUT_DIR = "./videomae-finetuned"

image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CKPT)
model = VideoMAEForVideoClassification.from_pretrained(MODEL_CKPT, label2id=LABEL2ID, id2label=ID2LABEL, ignore_mismatched_sizes=True)

train_ds, val_ds = prepare_dataset(
    train_df=train_df,
    val_df=val_df,
    image_processor=image_processor,
    model=model,
)

trainer = train_model(
    output_dir=OUTPUT_DIR,
    model=model,
    image_processor=image_processor,
    train_dataset=train_ds,
    val_dataset=val_ds,
    batch_size=4,
    num_epochs=100,
)

preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([22]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([22, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/mhmd7syn/gym-pose-videomae/f78c1dfd1c4941a293e30a647a573713

COMET INFO: Couldn't find a Git repository in '/kaggle/working' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewh

Epoch,Training Loss,Validation Loss,Accuracy
0,2.919100,2.945108,0.155452
1,1.903100,2.210098,0.422274
2,0.956700,1.360866,0.712297
3,0.422900,0.938635,0.809745
4,0.413300,0.688343,0.819026
5,0.156500,0.447504,0.888631
6,0.140600,0.425085,0.886311
7,0.176300,0.359956,0.911833
8,0.103900,0.222029,0.948956
9,0.148500,0.333945,0.916473


# Step 3. Inference and Evaluation

Run the trained model on a sample video or the validation set.

In [6]:
import math
import shutil
import uuid
import ffmpeg
from transformers import pipeline

def segment_video(video_length, chunk_size=10, overlap=3):
    if video_length < chunk_size + overlap:
        return [(0, video_length)]
    num_chunks = math.ceil((video_length - overlap) / (chunk_size - overlap))
    chunk_start = 0
    segments = []
    for i in range(num_chunks):
        chunk_end = min(chunk_start + chunk_size, video_length)
        segments.append((chunk_start, chunk_end))
        chunk_start += chunk_size - overlap
    return segments

def crop_video(video_file, start_time, end_time, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    output_filename = os.path.join(save_dir, f"{uuid.uuid4()}.mp4")
    video_length = int(float(ffmpeg.probe(video_file)["format"]["duration"]))
    if start_time == 0 and end_time == video_length:
        shutil.copy(video_file, output_filename)
        return output_filename
    (
        ffmpeg.input(video_file, ss=start_time, to=end_time)
        .output(output_filename)
        .run(quiet=True)
    )
    return output_filename

def infer_video(model_pool, video_path):
    # Use the best model from the trainer, or load from checkpoint
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pipe = pipeline(
        model=model_pool.model if model_pool else OUTPUT_DIR,
        task='video-classification',
        feature_extractor=image_processor,
        device=device,
    )
    
    video_length = int(float(ffmpeg.probe(video_path)["format"]["duration"]))
    segments = segment_video(video_length, chunk_size=5, overlap=1)
    
    preds = {}
    temp_dir = "./temp_crops"
    for start, end in segments:
         clip = crop_video(video_path, start, end, temp_dir)
         out = pipe(clip)
         # Aggregate scores
         for r in out:
             preds[r['label']] = preds.get(r['label'], 0) + r['score']
    
    # Cleanup
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
        
    return max(preds, key=preds.get)

# Test on a validation sample
sample_video = val_df.iloc[0]['path']
print(f"Testing on: {sample_video}")
prediction = infer_video(trainer, sample_video)
print(f"Prediction: {prediction}")

Device set to use cuda


Testing on: /kaggle/input/workoutfitness-video/leg extension/leg extension_1.MOV
Prediction: leg extension
